# 04.2 — Building a simple encoder

The registry ([04.1](04.1_architecture_string_dispatcher.ipynb)) turns `"GRU"` into a builder call. This notebook goes *inside* that builder — the `SimpleSequenceEncoder`, the project's Milestone-B encoder. It's a stack of near-identical blocks, and the interesting content is entirely in one detail: **the exact order of operations within a block** (Critical Note #27), which must match MATLAB layer-for-layer or checkpoints won't transfer.

This notebook covers:

* The stacked-block encoder structure.
* The block's fixed order: Transform → Dropout → Norm → Activation, and why order matters.
* Reading the MATLAB `cgg_constructSimpleCoder` and its Python twin side by side.
* A MATLAB naming bug the port faithfully reproduces (`SoftSign` → softplus).

**Prerequisites:** [02.6 nn.Module](../02_numpy_and_pytorch_basics/02.6_nn_module_vs_layergraph.ipynb), [04.1 the dispatcher](04.1_architecture_string_dispatcher.ipynb).

## Section 1 — What MATLAB does

`cgg_constructSimpleCoder` builds an encoder by stacking one block per hidden size:

```matlab
layers = [];
for k = 1:numel(HiddenSizes)
    switch Transform
        case 'GRU';  layers = [layers; gruLayer(HiddenSizes(k), 'OutputMode','sequence')];
        case 'LSTM'; layers = [layers; lstmLayer(HiddenSizes(k), 'OutputMode','sequence')];
        case 'Feedforward'; layers = [layers; fullyConnectedLayer(HiddenSizes(k))];
    end
    if Dropout > 0;         layers = [layers; dropoutLayer(Dropout)];      end
    if WantNormalization;   layers = [layers; layerNormalizationLayer];    end
    if ~isempty(Activation);layers = [layers; activationLayer(Activation)];end
end
```

The order inside the loop — **transform, then dropout, then norm, then activation** — is the whole game. Put layer-norm before dropout, or activation before norm, and you get a *different function*, so a MATLAB-trained checkpoint would produce different outputs in the port. Critical Note #27 pins this order.

## Section 2 — The Python concepts you need

### 2.1 — The block, built by hand

One block is a `Transform`, then three optional stages. Here it is with `nn.Sequential` to make the order literal:

In [ ]:
import torch
from torch import nn

def make_block(in_f, out_f, dropout=0.0, norm=False, activation=None):
    layers = [nn.Linear(in_f, out_f)]                 # 1. Transform (FC here)
    if dropout > 0:      layers.append(nn.Dropout(dropout))    # 2. Dropout
    if norm:             layers.append(nn.LayerNorm(out_f))    # 3. Norm
    if activation:       layers.append(activation)             # 4. Activation
    return nn.Sequential(*layers)

block = make_block(8, 16, dropout=0.5, norm=True, activation=nn.ReLU())
print(block)
print("forward:", tuple(block(torch.randn(2, 5, 8)).shape))

### 2.2 — Why the order is not negotiable

The four stages don't commute. A quick demonstration that swapping norm and activation changes the output distribution — same weights, same input, different function:

In [ ]:
torch.manual_seed(0)
x = torch.randn(1000, 8)
lin = nn.Linear(8, 16)
h = lin(x)

# order A (MATLAB): norm THEN activation
a = nn.ReLU()(nn.LayerNorm(16)(h))
# order B (swapped): activation THEN norm
b = nn.LayerNorm(16)(nn.ReLU()(h))

print(f"norm→relu:  mean {a.mean():+.3f}  std {a.std():.3f}  (values ≥ 0, then unnormalized)")
print(f"relu→norm:  mean {b.mean():+.3f}  std {b.std():.3f}  (renormalized AFTER clipping)")
print("different distributions ⇒ different function ⇒ checkpoints wouldn't transfer")

LayerNorm-then-ReLU leaves ReLU's output un-normalized (mean > 0, since negatives are clipped); ReLU-then-LayerNorm re-centers after clipping (mean ≈ 0). Neither is "wrong" in the abstract — but only ONE matches MATLAB, and matching is the requirement. This is why the port doesn't reach for a "cleaner" order: parity is the constraint.

### 2.3 — Stacking blocks: the encoder

In [ ]:
from neural_data_decoding.models.encoder import build_simple_encoder

enc = build_simple_encoder({
    "in_features": 8,
    "hidden_sizes": [32, 16, 4],       # THREE blocks: 8→32→16→4
    "transform": "GRU",
    "dropout": 0.5,
    "want_normalization": False,
    "activation": "",
})
print(f"{len(enc.blocks)} blocks, {sum(p.numel() for p in enc.parameters())} params")
for i, blk in enumerate(enc.blocks):
    print(f"  block {i}: {type(blk.transform_layer).__name__}, "
          f"dropout={type(blk.dropout).__name__}, norm={type(blk.norm).__name__}, "
          f"act={type(blk.activation).__name__}")

Each block's four slots are visible: the `GRU` transform, a real `Dropout` (rate 0.5), `Identity` for norm (off), `Identity` for activation (off — the production GRU choice). "Off" is an `nn.Identity`, not a missing layer — so the forward path is uniform and the block count matches `len(hidden_sizes)` exactly.

The GRU output note: `nn.GRU` returns `(sequence_output, final_hidden)`; the block keeps `[0]` — the per-timestep sequence, matching MATLAB's `OutputMode='sequence'`. The `W` (window) axis from [02.2](../02_numpy_and_pytorch_basics/02.2_array_axis_conventions.ipynb) is the sequence the GRU walks.

## Section 3 — The `neural_data_decoding` implementation

`_EncoderBlock.forward` — the four-line order, in production:

In [ ]:
from pathlib import Path
src = Path("../../src/neural_data_decoding/models/encoder.py").read_text().splitlines()
i = next(j for j, line in enumerate(src) if "def forward" in line and "_EncoderBlock" in "".join(src[max(0,j-40):j]))
# locate _EncoderBlock.forward specifically
for j, line in enumerate(src):
    if line.strip().startswith("def forward") and "Transform → Dropout → Norm → Activation" in "".join(src[j:j+3]):
        i = j; break
for k in range(i, min(i + 15, len(src))):
    print(f"{k + 1:4} | {src[k]}")

Things to spot:

* The forward is exactly `transform → dropout → norm → activation`, in that sequence, with a comment naming Critical Note #27.
* The GRU/LSTM branch takes `[0]` (sequence output); the Feedforward branch doesn't (a Linear returns a plain tensor).
* Dropout, norm, and activation are pre-built as either the real layer or `nn.Identity` in `__init__` — so `forward` has no `if dropout > 0` branching, keeping the hot path branch-free and the order un-editable.

And the MATLAB naming bug the port honors:

In [ ]:
i = next(j for j, line in enumerate(src) if line.startswith("def _make_activation"))
for k in range(i, min(i + 24, len(src))):
    print(f"{k + 1:4} | {src[k]}")

`'SoftSign'` instantiates `nn.Softplus`, NOT a softsign — because MATLAB's activation layer has that exact naming bug (Critical Note #37), and a checkpoint trained under MATLAB's mislabeled layer must load into the same behavior. This is parity discipline at its sharpest: **reproduce the bug, document it loudly, and steer new configs away.** Faithfulness to a reference sometimes means copying its mistakes.

## Section 4 — Hands-on exercises

### Exercise 4.1 — count the block slots

Build a Feedforward encoder with `hidden_sizes=[10, 10]`, `dropout=0.3`, `want_normalization=True`, `activation="ReLU"`. How many *real* (non-Identity) sub-layers total? Predict, then inspect.

In [ ]:
# Your attempt here


In [ ]:
# Reveal:
enc = build_simple_encoder({"in_features": 6, "hidden_sizes": [10, 10],
    "transform": "Feedforward", "dropout": 0.3, "want_normalization": True, "activation": "ReLU"})
real = 0
for blk in enc.blocks:
    for layer in (blk.transform_layer, blk.dropout, blk.norm, blk.activation):
        if not isinstance(layer, nn.Identity):
            real += 1
print(f"{len(enc.blocks)} blocks × 4 real slots = {real} real sub-layers")

### Exercise 4.2 — the empty encoder

What does `hidden_sizes=[]` produce? Build it and pass a tensor through — predict the output shape first. (This is the "no encoder" case the Logistic-Regression model uses.)

In [ ]:
# Reveal:
identity_enc = build_simple_encoder({"in_features": 8, "hidden_sizes": [],
    "transform": "GRU", "dropout": 0.0, "want_normalization": False, "activation": ""})
x = torch.randn(2, 5, 8)
out = identity_enc(x)
print(f"{len(identity_enc.blocks)} blocks — output shape {tuple(out.shape)} (unchanged: identity)")

## Section 5 — Common errors

### Checkpoint loads but the model behaves differently

Block-order drift (§2.2) — a "cleaner" reordering breaks parity. The order is `transform → dropout → norm → activation`, period. Don't optimize it.

### `RuntimeError` on the first forward: size mismatch

`in_features` must equal the input's last-axis size, and each block's output feeds the next — a `hidden_sizes` typo (e.g. `[16, 4]` when the classifier expects a 250-d latent) surfaces here or at the classifier boundary.

### GRU/LSTM output looks wrong (only the last timestep)

You grabbed the final hidden state (`[1]`) instead of the sequence (`[0]`). The encoder wants the per-timestep sequence to preserve the `W` axis.

### `SoftSign` doesn't behave like softsign

By design (§3) — it's softplus, matching MATLAB's bug. Use `''` (no activation) or a correctly-named one in new configs.

### `ValueError: Unsupported transform / activation`

The block only accepts `Feedforward`/`GRU`/`LSTM` and a fixed activation set — the constructor validates up front (a good example of failing loudly at construction rather than mysteriously at forward).

## Section 6 — Further reading

- [nn.GRU / nn.LSTM docs](https://pytorch.org/docs/stable/generated/torch.nn.GRU.html) — the (output, hidden) return contract.
- [LayerNorm docs](https://pytorch.org/docs/stable/generated/torch.nn.LayerNorm.html) — what "normalize" means here.
- [`src/neural_data_decoding/models/encoder.py`](../../src/neural_data_decoding/models/encoder.py) — the full builder + block, with the Critical Note citations.

Next up: **[04.3 RNN building blocks](04.3_rnn_building_blocks.ipynb)** — GRU and LSTM internals, `batch_first`, and hidden-state handling.